In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [3]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/T47D.csv")

X_well = df.drop(columns=['Metadata_well', 'phase'])
y_well = df['Metadata_well']

X_train_well, X_test_well, y_train_well, y_test_well = train_test_split(X_well, y_well, test_size=0.2, random_state=949, stratify=y)

In [4]:
# Define MLP model
mlp = MLPClassifier(max_iter=1000, random_state=949)

In [7]:
# Hyperparameter tuning
param_grid = {
    'hidden_layer_sizes': [
        (22,),         # 1 hidden layer
        (11,),        # 1 hidden layer
        (22, 11),     # 2 hidden layers
        (22, 22),       # 2 hidden layers
        (22,22,22)   # 3 hidden layers
    ]
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    cv=10,
    scoring='accuracy',
    n_jobs=-1
)

# Fit model
grid_search.fit(X_train_well, y_train_well)

# Output best parameters and best accuracy
print("Best parameters:", grid_search.best_params_)

Best parameters: {'hidden_layer_sizes': (22, 22)}


In [8]:
# Convert the cv_results_ dictionary to a DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)

# Select and display relevant columns
print(
    results_df[
        [
            'param_hidden_layer_sizes',
            'mean_test_score',
            'std_test_score',
            'rank_test_score'
        ]
    ].sort_values(by='rank_test_score')
)

  param_hidden_layer_sizes  mean_test_score  std_test_score  rank_test_score
3                 (22, 22)         0.744443        0.007346                1
4             (22, 22, 22)         0.741032        0.006161                2
2                 (22, 11)         0.740606        0.007866                3
0                    (22,)         0.735199        0.005549                4
1                    (11,)         0.710044        0.006825                5


In [9]:
# Retrain with best parameters
mlp = MLPClassifier(max_iter=1000, random_state=949, hidden_layer_sizes=(22,22))
mlp.fit(X_train_well, y_train_well)

# Predictions
y_train_pred = mlp.predict(X_train_well)
y_test_pred = mlp.predict(X_test_well)

In [10]:
# Evaluation
print("=== Training Set ===")
print("Overall Accuracy:", accuracy_score(y_train_well, y_train_pred))

print("\n=== Test Set ===")
print("Overall Accuracy:", accuracy_score(y_test_well, y_test_pred))

=== Training Set ===
Overall Accuracy: 0.7570008333171838

=== Test Set ===
Overall Accuracy: 0.7468413301294473


In [6]:
# save results
# === Load existing results ===
results_df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Bar Plot/classification_results.csv", index_col=0)

# === Compute accuracy ===
from sklearn.metrics import accuracy_score

overall_acc = accuracy_score(y_test, y_test_pred)

df_test = pd.DataFrame({'true': y_test, 'pred': y_test_pred})
acc_per_phase = df_test.groupby('true').apply(lambda x: accuracy_score(x['true'], x['pred']))

# === Insert values ===
model_name = "MLP (full)"
results_df.loc[model_name, 'Overall'] = overall_acc

# Set per-phase accuracies
for phase in ['G0', 'G1', 'G2', 'S']:
    if phase in acc_per_phase.index:
        results_df.loc[model_name, phase] = acc_per_phase[phase]

# === Save updated file ===
results_df.to_csv("/Users/mariahloehr/IICD/IICD/Bar Plot/classification_results.csv")

/var/folders/1s/bvxr71hj0hqgyk_jk6k7wkm80000gn/T/ipykernel_44609/2108438782.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  acc_per_phase = df_test.groupby('true').apply(lambda x: accuracy_score(x['true'], x['pred']))
